# 6. Évaluation du modèle YOLOv9-C sur le jeu de test (`val_dual.py`)

Ce notebook charge le meilleur checkpoint `best.pt` issu de l'entraînement, exécute l'évaluation sur le jeu de **test** (`405` images), extrait les métriques globales et par classe, analyse l'impact du déséquilibre de classes et compare les paramètres d'évaluation à ceux de visualisation.

In [ ]:
from pathlib import Path
from collections import Counter
import subprocess
import re
import sys
import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

YOLO_DIR = Path("../yolov9").resolve()
DATA_DIR = Path("../trafic_data").resolve()
DATA_YAML = DATA_DIR / "data_1.yaml"
VAL_SCRIPT = YOLO_DIR / "val_dual.py"
TRAIN_RUN_NAME = "yolov9c_vehicles_e10"
BEST_PT = YOLO_DIR / "runs/train" / TRAIN_RUN_NAME / "weights/best.pt"
EVAL_NAME = "yolov9c_vehicles_test"
EVAL_DIR = YOLO_DIR / "runs/val" / EVAL_NAME

HAS_CUDA = torch.cuda.is_available()
DEVICE = "0" if HAS_CUDA else "cpu"
print(f"Device détecté : {DEVICE} | Checkpoint ciblé : {BEST_PT} (existe={BEST_PT.exists()})")

val_cmd = [
    sys.executable,
    str(VAL_SCRIPT),
    "--data", str(DATA_YAML),
    "--weights", str(BEST_PT),
    "--task", "test",
    "--img", "640",
    "--batch-size", "8" if HAS_CUDA else "4",
    "--conf-thres", "0.001",
    "--iou-thres", "0.7",
    "--device", DEVICE,
    "--verbose",
    "--project", str(YOLO_DIR / "runs/val"),
    "--name", EVAL_NAME,
    "--exist-ok",
]
print("Commande d'évaluation prête :\n", " ".join(val_cmd))

In [ ]:
# Exécuter l'évaluation sur le jeu de test (nécessite best.pt généré à l'étape 5)
assert BEST_PT.exists(), (
    f"Checkpoint introuvable : {BEST_PT}. "
    "Exécutez d'abord l'entraînement dans trainYolov9.ipynb (sur GPU) pour générer best.pt."
)

proc = subprocess.run(
    val_cmd,
    cwd=str(YOLO_DIR),
    capture_output=True,
    text=True,
)
eval_output = (proc.stdout or "") + "\n" + (proc.stderr or "")
print(eval_output)
print("Code de retour val_dual.py :", proc.returncode)

In [ ]:
# Extraire les métriques globales et par classe, et les comparer avec la taille des classes dans train/test
with open(DATA_YAML, "r", encoding="utf-8") as f:
    names = yaml.safe_load(f)["names"]

counts = {}
for split in ["train", "test"]:
    c = Counter()
    for lbl in (DATA_DIR / split / "labels").glob("*.txt"):
        for line in lbl.read_text(encoding="utf-8").splitlines():
            if line.strip():
                c[int(line.split()[0])] += 1
    counts[split] = c

if "eval_output" in globals() and eval_output.strip():
    # Parser les lignes de format '%22s%11i%11i%11.3g%11.3g%11.3g%11.3g' affichées par val_dual.py
    pattern = re.compile(
        r"^\s*(?P<classe>.+?)\s+(?P<images>\d+)\s+(?P<instances>\d+)\s+"
        r"(?P<precision>[0-9.]+)\s+(?P<recall>[0-9.]+)\s+(?P<map50>[0-9.]+)\s+(?P<map50_95>[0-9.]+)\s*$"
    )
    rows = []
    for line in eval_output.splitlines():
        m = pattern.match(line)
        if m:
            d = m.groupdict()
            rows.append({
                "classe": d["classe"].strip(),
                "images": int(d["images"]),
                "instances_test": int(d["instances"]),
                "Precision": float(d["precision"]),
                "Recall": float(d["recall"]),
                "mAP@50": float(d["map50"]),
                "mAP@50-95": float(d["map50_95"]),
            })

    df_eval = pd.DataFrame(rows)
    if not df_eval.empty:
        name_to_id = {n: i for i, n in enumerate(names)}
        df_eval["instances_train"] = df_eval["classe"].map(
            lambda cl: sum(counts["train"].values()) if cl == "all" else counts["train"].get(name_to_id.get(cl, -1), 0)
        )
        df_global = df_eval[df_eval["classe"] == "all"]
        df_cls = df_eval[df_eval["classe"] != "all"].sort_values("mAP@50", ascending=False)

        print("=== Métriques globales (test) ===")
        print(df_global.to_string(index=False))
        print("\n=== Résultats par classe (triés par mAP@50) ===")
        print(df_cls.to_string(index=False))
        print("\nTop 5 classes mieux détectées :", df_cls.head(5)["classe"].tolist())
        print("Top 5 classes les plus faibles   :", df_cls.tail(5)["classe"].tolist())

        ax = df_cls.set_index("classe")[["mAP@50", "mAP@50-95"]].plot(kind="bar", figsize=(13, 5))
        ax.set_title("Comparaison mAP@50 et mAP@50-95 par classe (jeu de test)")
        ax.set_ylabel("Score")
        plt.xticks(rotation=75)
        plt.tight_layout()
        plt.show()
else:
    print("Aucune sortie d'évaluation disponible : exécutez d'abord la cellule d'évaluation ci-dessus.")

In [ ]:
# Afficher les graphiques générés par val_dual.py (matrice de confusion, PR_curve, F1_curve)
for img_name in ["confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    p = EVAL_DIR / img_name
    if p.exists():
        with Image.open(p) as im:
            plt.figure(figsize=(10, 8))
            plt.imshow(im)
            plt.axis("off")
            plt.title(img_name)
            plt.show()
    else:
        print(f"[En attente d'évaluation] {p}")

### Synthèse : Métriques d'évaluation, limites du dataset et différence avec la visualisation

1. **Définition des métriques :**
   - **Precision :** proportion de prédictions correctes parmi toutes les détections émises (`TP / (TP + FP)`). Mesure la fiabilité (peu de faux positifs).
   - **Recall :** proportion de véhicules réels effectivement détectés (`TP / (TP + FN)`). Mesure la couverture (peu d'objets manqués).
   - **mAP@50 :** moyenne sur toutes les classes de l'*Average Precision* calculée à un seuil `IoU ≥ 0.50`.
   - **mAP@50–95 :** moyenne de l'AP sur 10 seuils d'IoU de `0.50` à `0.95` (pas de `0.05`). Beaucoup plus exigeante sur la précision géométrique des boîtes englobantes.

2. **Impact du déséquilibre de classes :**
   - Les classes fréquentes (`car`: 3 848 train / 784 test, `rickshaw`: 2 836 / 500, `bus`: 2 465 / 444, `three wheelers -CNG-`: 2 281 / 453, `motorbike`: 1 660 / 289) fournissent suffisamment d'exemples variés pour bien généraliser.
   - Les classes rares (`garbagevan`: 3 train / 0 test — impossible à évaluer sur `test`, `policecar`: 23 / 8, `scooter`: 30 / 7, `army vehicle`: 35 / 8, `taxi`: 36 / 5, `ambulance`: 54 / 16, `minibus`: 77 / 16) ont très peu d'exemples d'entraînement et de test, ce qui rend leurs métriques instables ou faibles et augmente les confusions avec des classes proches (`car`, `van`, `truck`, `motorbike`).

3. **Paramètres d'évaluation (`val_dual.py`) vs paramètres de visualisation (`detect_dual.py`) :**
   - **Évaluation (`val_dual.py`) :** utilise un seuil de confiance très bas (`--conf-thres 0.001`) et un NMS plus tolérant (`--iou-thres 0.7`, `--max-det 300`) afin de balayer toute la courbe Precision–Recall et calculer une mAP non biaisée.
   - **Visualisation / inférence (`detect_dual.py`) :** utilise un seuil de confiance élevé (`--conf-thres 0.25`) et un NMS plus strict (`--iou-thres 0.45`) pour n'afficher sur les images que les détections fiables et éviter les boîtes superposées.